# 03 · Normalization and Highly Variable Genes

**Dataset:** GSM9936742 · Human Femoral Head · Visium HD (56 µm aggregated bins, inferred from barcodes)

**Input:** `results/adata/02_qc.h5ad`  
**Output:** `results/adata/03_normalized.h5ad`

This notebook keeps raw counts in `layers["counts"]`, applies per-bin library-size normalization and log1p, identifies HVGs without dropping genes, creates QC visualizations, and verifies the saved output. **QC filtering was not applied in Step 02**; review the QC figures before interpreting downstream biological results.


## 0 · Imports and reproducible paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data").is_dir() and (p / "results").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside visium_hd_tutorial or its notebooks directory")
INPUT = ROOT / "results" / "adata" / "02_qc.h5ad"
OUTPUT = ROOT / "results" / "adata" / "03_normalized.h5ad"
FIGURES = ROOT / "results" / "figures"
TABLES = ROOT / "results" / "tables"
for directory in (OUTPUT.parent, FIGURES, TABLES):
    directory.mkdir(parents=True, exist_ok=True)
assert INPUT.is_file(), f"Missing input: {INPUT}. Run 02_quality_control.ipynb first."
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"))
print("Project root:", ROOT)
print("Input:", INPUT.relative_to(ROOT))

## 1 · Load QC AnnData and validate raw counts

In [ ]:
adata = sc.read_h5ad(INPUT)
assert sparse.issparse(adata.X), "Expected a sparse count matrix"
assert adata.obs_names.is_unique and adata.var_names.is_unique
assert "spatial" in adata.obsm and adata.obsm["spatial"].shape == (adata.n_obs, 2)
assert "total_counts" in adata.obs and "n_genes_by_counts" in adata.obs
assert adata.n_obs > 0 and adata.n_vars > 0
assert np.isfinite(adata.X.data).all()
assert np.all(adata.X.data >= 0), "Counts cannot be negative"
assert np.allclose(adata.X.data, np.rint(adata.X.data)), "Input is not integer-like raw counts"
print(adata)
print("QC filtering status:", adata.uns.get("qc_filtering", "Check Step 02 metadata"))
print("Zero-count bins:", int(np.sum(np.asarray(adata.X.sum(axis=1)).ravel() == 0)))

## 2 · Preserve raw counts and calculate normalization factors

The raw counts are preserved in a sparse layer. Zero-count bins remain zero; they are flagged for downstream review.

In [ ]:
# Preserve the full raw count matrix (not normalized or log-transformed).
adata.layers["counts"] = adata.X.copy()
raw_totals = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()
adata.obs["total_counts_before_normalization"] = raw_totals
TARGET_SUM = 1e4
N_TOP_GENES = min(2000, adata.n_vars)
print("Total counts before normalization (median):", float(np.median(raw_totals)))
print("Normalization target:", TARGET_SUM)
print("HVG target:", N_TOP_GENES)

## 3 · Library-size normalization and log1p

In [ ]:
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
normalized_totals = np.asarray(adata.X.sum(axis=1)).ravel()
positive = raw_totals > 0
assert np.allclose(normalized_totals[positive], TARGET_SUM, rtol=1e-4)
assert np.all(normalized_totals[~positive] == 0)
sc.pp.log1p(adata)
assert sparse.issparse(adata.X), "Expected log-normalized matrix to remain sparse"
assert np.isfinite(adata.X.data).all()
print("Normalized and log1p-transformed:", adata.shape)
print("Raw counts preserved in adata.layers['counts']:", adata.layers["counts"].shape)

## 4 · Highly variable gene selection

Use the `seurat` flavor on **log1p-normalized** data. Keep all genes; `adata.var["highly_variable"]` is a selection flag for the next notebook.

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    n_top_genes=N_TOP_GENES,
    subset=False,
    inplace=True,
)
assert "highly_variable" in adata.var
n_hvg = int(adata.var["highly_variable"].sum())
assert n_hvg > 0
print("HVGs selected:", n_hvg)
print("All genes retained:", adata.n_vars)
hvg_table = adata.var.loc[adata.var["highly_variable"], ["highly_variable", "means", "dispersions", "dispersions_norm"]].copy()
hvg_table.index.name = "gene"
hvg_table.to_csv(TABLES / "03_highly_variable_genes.csv")
print("Saved:", (TABLES / "03_highly_variable_genes.csv").relative_to(ROOT))

## 5 · Figures

Plots summarize per-bin library sizes and normalized expression. These are **not** registered tissue-image overlays.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Before normalization
axes[0].hist(np.log1p(raw_totals), bins=60)
axes[0].set(
    title="Raw library sizes",
    xlabel="log1p(total UMI per bin)",
    ylabel="Number of bins",
)

# After normalization
# A nearly constant library size should not be plotted
# using automatically generated histogram bins.
values = np.asarray(normalized_totals[positive], dtype=float).ravel()
values = values[np.isfinite(values)]

if values.size == 0:
    axes[1].text(
        0.5, 0.5, "No nonzero bins",
        ha="center", va="center",
        transform=axes[1].transAxes,
    )
elif np.ptp(values) < max(1e-6, abs(np.mean(values)) * 1e-8):
    axes[1].hist(
        values,
        bins=np.linspace(np.mean(values) - 1, np.mean(values) + 1, 21),
    )
else:
    axes[1].hist(values, bins=40)

axes[1].set(
    title="After library-size normalization (nonzero bins)",
    xlabel="Total normalized counts",
    ylabel="Number of bins",
)

fig.tight_layout()
fig.savefig(
    FIGURES / "03_library_size_normalization.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

## 6 · Record processing metadata and save

In [ ]:
adata.uns["normalization_pipeline"] = {
    "method": "scanpy.pp.normalize_total + log1p",
    "target_sum": float(TARGET_SUM),
    "hvg_flavor": "seurat",
    "hvg_n_top_genes_requested": int(N_TOP_GENES),
    "hvg_n_selected": int(n_hvg),
    "raw_count_layer": "counts",
    "input": "results/adata/02_qc.h5ad",
}
adata.write_h5ad(OUTPUT)
print("Saved:", OUTPUT.relative_to(ROOT))
print("File size (MB):", round(OUTPUT.stat().st_size / (1024 ** 2), 2))

## 7 · Reload test (pipeline handoff)

In [ ]:
check = sc.read_h5ad(OUTPUT)
assert check.shape == adata.shape
assert check.obs_names.equals(adata.obs_names)
assert check.var_names.equals(adata.var_names)
assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
assert np.array_equal(check.layers["counts"].data, adata.layers["counts"].data)
assert "highly_variable" in check.var
assert int(check.var["highly_variable"].sum()) == n_hvg
assert np.isfinite(check.X.data).all()
print("PASS — 03_normalization completed")
print("AnnData:", check.shape)
print("HVGs:", n_hvg)
print("Raw count layer:", check.layers["counts"].shape)
print("Output:", OUTPUT.relative_to(ROOT))

## Next: `04_clustering.ipynb`

Read `results/adata/03_normalized.h5ad`. Use the HVG flag for PCA, construct a neighborhood graph, compute UMAP and Leiden clusters. **Do not treat 56 µm bins as individual cells.** Image registration remains unverified; use coordinate-only spatial plots until a trusted transform is available.
